# XAI-SDN Quickstart Notebook

This notebook walks through the full XAI-SDN pipeline:
1. Generate synthetic DDoS traffic data
2. Compute entropy features
3. Train the Random Forest classifier
4. Evaluate performance
5. Inspect SHAP attributions

> **No dataset download required** — uses synthetic data by default.
> To use real CIC-DDoS2019 data, replace `load_synthetic_data()` with `load_real_data()`.

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

## 1. Entropy Feature Demo

Shannon entropy H(X) = -Σ p(xᵢ) log₂ p(xᵢ) measures diversity.
Low entropy → concentrated (attack); High entropy → diverse (normal traffic).

In [ ]:
import math
from features.entropy import shannon_entropy, EntropyFeatureExtractor, ENTROPY_FEATURE_NAMES

# Demonstrate entropy properties
print('H([a,a,a,a]) =', shannon_entropy(['a','a','a','a']), ' ← single value = 0 bits')
print('H([a,b])     =', shannon_entropy(['a','b']),       ' ← two equal = 1 bit')
print('H([a,b,c,d]) =', round(shannon_entropy(['a','b','c','d']),4), ' ← four equal = 2 bits')
print()

# Simulate UDP flood vs. benign traffic
extractor = EntropyFeatureExtractor(window_size=100)

benign_flows = [
    {'src_ip': f'10.0.0.{i%50}', 'dst_ip': f'10.0.1.{i%20}',
     'dst_port': [80,443,22,53,8080][i%5], 'protocol': [6,17][i%2],
     'pkt_len_mean': 200+i*3, 'iat_mean': 500+i*10, 'tcp_flags': i%8, 'ttl': 64}
    for i in range(100)
]
ddos_flows = [
    {'src_ip': '10.0.1.100', 'dst_ip': '10.0.0.1',
     'dst_port': 53, 'protocol': 17,
     'pkt_len_mean': 64.0, 'iat_mean': 100.0, 'tcp_flags': 0, 'ttl': 64}
    for _ in range(100)
]

extractor.reset()
for f in benign_flows: benign_feats = extractor.update_and_compute(f)
extractor.reset()
for f in ddos_flows:   ddos_feats   = extractor.update_and_compute(f)

df_compare = pd.DataFrame({'Benign': benign_feats, 'DDoS-UDP': ddos_feats})
print(df_compare.to_string())

In [ ]:
# Plot entropy comparison
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(ENTROPY_FEATURE_NAMES))
width = 0.35
ax.bar(x - width/2, [benign_feats[k] for k in ENTROPY_FEATURE_NAMES], width,
       label='Benign', color='steelblue', alpha=0.85)
ax.bar(x + width/2, [ddos_feats[k] for k in ENTROPY_FEATURE_NAMES], width,
       label='DDoS-UDP', color='crimson', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(ENTROPY_FEATURE_NAMES, rotation=30, ha='right')
ax.set_ylabel('Entropy (bits)'); ax.set_title('Entropy Features: Benign vs DDoS-UDP')
ax.legend(); plt.tight_layout(); plt.show()

## 2. Generate Synthetic Dataset and Train Model

In [ ]:
from model.train import load_synthetic_data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
import time

# Load synthetic data
X, y, le = load_synthetic_data(n_samples=10000, random_state=42)
print(f'Dataset: {X.shape[0]} samples × {X.shape[1]} features')
print(f'Classes: {list(le.classes_)}')
print(f'\nClass distribution:')
unique, counts = np.unique(y, return_counts=True)
for cls_id, cnt in zip(unique, counts):
    print(f'  {le.classes_[cls_id]:20s}  {cnt:5d}  ({cnt/len(y)*100:.1f}%)')

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'\nTrain: {X_train_s.shape[0]}, Test: {X_test_s.shape[0]}')

In [ ]:
# Train Random Forest
clf = RandomForestClassifier(
    n_estimators=200, max_features='sqrt',
    class_weight='balanced', random_state=42, n_jobs=-1
)
t0 = time.perf_counter()
clf.fit(X_train_s, y_train)
train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = clf.predict(X_test_s)
inf_time = time.perf_counter() - t0

macro_f1 = f1_score(y_test, y_pred, average='macro')
lat_ms   = inf_time / len(X_test) * 1000
tput     = len(X_test) / inf_time

print(f'Train time:   {train_time:.1f}s')
print(f'Macro F1:     {macro_f1:.4f}')
print(f'Latency:      {lat_ms:.3f} ms/flow')
print(f'Throughput:   {tput:,.0f} flows/s')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 3. Feature Importance

In [ ]:
from features.cicflowmeter import CIC_FEATURE_NAMES
from features.entropy import ENTROPY_FEATURE_NAMES

all_feature_names = CIC_FEATURE_NAMES + ENTROPY_FEATURE_NAMES
importances = clf.feature_importances_

imp_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(20)

colors = ['coral' if f.startswith('H_') else 'steelblue' for f in imp_df['feature']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color=colors[::-1])
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 20 Features — RF Feature Importance\n(coral = entropy features)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

ent_total = importances[80:].sum()
print(f'\nEntropy features (H_*) total importance: {ent_total:.4f} ({ent_total*100:.1f}%)')

## 4. SHAP Attribution (requires `pip install shap`)

If SHAP is not installed, this cell will show a warning and skip gracefully.

In [ ]:
try:
    import shap
    from explainability.shap_explainer import SHAPExplainer

    explainer = SHAPExplainer(clf, all_feature_names)

    # Pick a DDoS-UDP flow from the test set
    ddos_label_id = list(le.classes_).index('DDoS-UDP')
    ddos_idx = np.where(y_test == ddos_label_id)[0][0]
    x_sample = X_test_s[ddos_idx]
    pred_class = int(clf.predict(x_sample.reshape(1,-1))[0])
    confidence = float(clf.predict_proba(x_sample.reshape(1,-1))[0].max())

    ranked = explainer.explain_flow_ranked(x_sample, predicted_class_idx=pred_class, top_k=15)

    print(f'Predicted: {le.classes_[pred_class]} (confidence={confidence:.2%})')
    print(f'\nTop 15 SHAP features:')
    print(f'{"Feature":<30} {"SHAP Value":>12}')
    print('-' * 45)
    for item in ranked:
        marker = ' ← entropy' if item['feature'].startswith('H_') else ''
        print(f"{item['feature']:<30} {item['shap_value']:+.6f}{marker}")

    # Plot waterfall
    from explainability.visualizations import plot_local_waterfall
    attribution = {item['feature']: item['shap_value'] for item in ranked}
    plot_local_waterfall(
        attribution, le.classes_[pred_class], confidence, top_k=12
    )

except ImportError:
    print('⚠️  SHAP not installed. Run: pip install shap')
    print('   SHAP explanation skipped.')

## 5. Next Steps

- `model/train.py --data-dir data/raw` — train on real CIC-DDoS2019
- `model/evaluate.py --run-shap` — full evaluation with global SHAP summary
- `model/ablation.py` — reproduce Table 4 from the paper
- `uvicorn api.main:app --reload` — start the REST API
- `streamlit run dashboard/app.py` — open the SOC dashboard